<a href="https://colab.research.google.com/github/locbp-uzh/biopipelines/blob/main/ExamplePipelines/notebooks/iterative_binding_optimization_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Iterative Binding Optimization

**BioPipelines example** — iterative directed evolution loop that uses LigandMPNN to propose sequence variants and Boltz2 to evaluate their binding affinity, keeping only the best candidate each cycle.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [1]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[all]"
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /usr/local bin/micromamba
!micromamba create -f Environments/biopipelines.yaml -y

Cloning into 'biopipelines'...
remote: Enumerating objects: 7076, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 7076 (delta 168), reused 170 (delta 138), pack-reused 6829 (from 3)
Receiving objects: 100% (7076/7076), 13.43 MiB | 33.88 MiB/s, done.
Resolving deltas: 100% (5253/5253), done.
/content/biopipelines
Obtaining file:///content/biopipelines
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.1 MB/s eta 0:00:00
  Building editable for biopipelines (pyproject.toml) ... done
  Created wheel for biopipelines: filename=biopipelines-1.1.0-0.editable

In [2]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2
from biopipelines.ligand_mpnn import LigandMPNN
from biopipelines.mutation_profiler import MutationProfiler

with Pipeline(project="Setup", job="InstallTools"):
    Boltz2.install()
    LigandMPNN.install()
    MutationProfiler.install()


Running Boltz2_installation (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
=== Installing Boltz2 ===
conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache


Transaction

  Prefix: /root/.local/share/mamba/envs/Boltz2Env

  Updating specs:

   - python=3.11


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge     Cached
  + bzip2                 1.0.8  hda65f42_9            conda-forge     Cached
  + ca-certificates   2026.2.25  hbd8a1cb_0            

## Cell 3: Iterative Binding Optimization Pipeline

Starting from the NocT protein (PDB: 5OT9) and Histopine as ligand, this pipeline runs 5 cycles of:
1. Identify binding-pocket residues within 5 Å of the ligand
2. Generate 1000 sequence variants with LigandMPNN
3. Build a mutation frequency profile
4. Compose 3 candidate sequences by weighted random sampling (max 3 mutations)
5. Predict binding affinities with Boltz2 and keep the best candidate

In [3]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2
from biopipelines.ligand_mpnn import LigandMPNN
from biopipelines.distance_selector import DistanceSelector
from biopipelines.mutation_profiler import MutationProfiler
from biopipelines.mutation_composer import MutationComposer
from biopipelines.panda import Panda

with Pipeline(project="NocT", job="IterativeBindingOptimization"):
    protein = PDB("5OT9")
    ligand = Ligand("Histopine")
    original = Boltz2(proteins=protein,
                      ligands=ligand)

    current_best = Panda(tables=original.tables.affinity,
                         pool=original)

    for cycle in range(3):
        Suffix(f"Cycle{cycle+1}")
        pocket = DistanceSelector(structures=current_best,
                                  ligand="LIG",
                                  distance=5)
        variants = LigandMPNN(structures=current_best,
                              ligand="LIG",
                              num_sequences=1000,
                              redesigned=pocket.tables.selections.within)
        profile = MutationProfiler(original=current_best,
                                   mutants=variants)
        candidates = MutationComposer(frequencies=profile.tables.absolute_frequencies,
                                      num_sequences=3,
                                      mode="weighted_random",
                                      max_mutations=3)
        predicted = Boltz2(proteins=candidates,
                           ligands=ligand)
        current_best = Panda(tables=[current_best.tables.result,
                                     predicted.tables.affinity],
                             operations=[Panda.concat(add_source=True),
                                          Panda.sort("affinity_probability_binary",
                                                     ascending=False),
                                          Panda.head(1)],
                             pool=[current_best,
                                   predicted])
current_best

  5OT9 not found locally, will download from RCSB

Running PDB (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
Fetching 1 structures
Convert: keep as-is (pdb|cif)
PDB IDs: 5OT9
Custom IDs: 5OT9
Priority: PDBs/ -> RCSB download
Output folder: /content/biopipelines/tests/NocT/IterativeBindingOptimization_001/001_PDB
Fetching 1 structures (keep as-is (pdb|cif))
Priority: PDBs/ -> RCSB download
Water molecules will be removed

[1/1] Processing 5OT9 -> 5OT9
5OT9 not found locally, downloading from RCSB
Cached to PDBs/ folder: /content/biopipelines/PDBs/5OT9.pdb
  Found 1 ligand(s) in structure: AOZ
  Found SMILES for AOZ: CC(C(=O)O)NC(Cc1c[nH]cn1)C(=O)O...
Successfully downloaded 5OT9 as 5OT9: 678942 bytes (rcsb_download (PDB))

Successful fetches saved: /content/biopipelines/tests/NocT/IterativeBindingOpt

StandardizedOutput({'structures': DataStream(name='structures', format='pdb', items=1, files=1, map_table=set), 'sequences': DataStream(name='sequences', format='csv', items=1, files=0, map_table=set), 'compounds': DataStream(name='compounds', format='csv', items=1, files=1, map_table=set), 'msas': DataStream(name='msas', format='csv', items=1, files=1, map_table=set), 'tables': {'result': TableInfo(name='result', path='/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3/concat_sort_head.csv', columns=['id', 'input_file', 'affinity_pred_value', 'affinity_probability_binary', 'source_table'], count=variable), 'missing': TableInfo(name='missing', path='/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3/missing.csv', columns=['id', 'removed_by', 'cause'], count=variable), 'structures': TableInfo(name='structures', path='/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3/structures_map.csv', columns=['id', 'file'], count=1), 'confidence': TableInfo(name='confidence', path='/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3/confidence_scores.csv', columns=['id', 'input_file', 'confidence_score', 'ptm', 'iptm', 'complex_plddt', 'complex_iplddt'], count=1), 'sequences': TableInfo(name='sequences', path='/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3/sequences.csv', columns=['id', 'sequence'], count=1), 'msas': TableInfo(name='msas', path='/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3/msas.csv', columns=['id', 'sequences.id', 'sequence', 'msa_file'], count=1), 'affinity': TableInfo(name='affinity', path='/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3/affinity_scores.csv', columns=['id', 'input_file', 'affinity_pred_value', 'affinity_probability_binary'], count=1), 'compounds': TableInfo(name='compounds', path='/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3/compounds.csv', columns=['id', 'format', 'smiles', 'ccd'], count=1)}, 'output_folder': '/content/biopipelines/tests/NocT/IterativeBindingOptimization_001/022_Panda_Cycle3'})